# Search Quality Baseline
Sample issues from DB → generate realistic user queries via Claude → run cosine similarity search → measure Hit@3 and MRR.

In [2]:
%pip install psycopg2-binary python-dotenv openai anthropic numpy -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import numpy as np
import psycopg2
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic

load_dotenv()

DATABASE_URL = os.getenv('DATABASE_URL').split('?')[0]
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
anthropic_client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

def get_conn():
    return psycopg2.connect(DATABASE_URL, sslmode='require')

def embed(text):
    return openai_client.embeddings.create(model='text-embedding-3-small', input=text[:8000]).data[0].embedding

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [4]:
# Sample N issues that have a non-empty body and resolution_text
N = 30

conn = get_conn()
cur = conn.cursor()
cur.execute("""
    SELECT i.numeric_id, i.issue_number, i.title, i.body, i.resolution_text, l.name as library
    FROM issues i
    JOIN libraries l ON i.library_id = l.id
    WHERE i.body IS NOT NULL AND i.body != ''
      AND i.resolution_text IS NOT NULL AND i.resolution_text != ''
    ORDER BY RANDOM()
    LIMIT %s
""", (N,))
sampled = cur.fetchall()
cur.close()
conn.close()

print(f'Sampled {len(sampled)} issues')
print(f'Example: [{sampled[0][5]}] #{sampled[0][1]} — {sampled[0][2][:80]}')

Sampled 30 issues
Example: [flask] #1232 — The 'download as PDF and zipped HTML' links on flask.pocoo.org are broken. 


In [5]:

# Generate a realistic agent query for each sampled issue using Claude Sonnet 4.6
SYSTEM_CONTEXT = """You are simulating Claude Code — an AI coding agent that helps developers debug and write code.

You have access to an MCP tool called search_bugs. Here is its description:
"Search indexed GitHub issues for bug solutions matching an error message or symptom. Use the exact error string or a short symptom description as the query."

You call this tool when a developer's code throws an error or behaves unexpectedly and you want to find a known solution from a similar GitHub issue. Your query is what you pass to the `query` argument of search_bugs."""

def generate_query(title, body, resolution):
    msg = anthropic_client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=150,
        system=SYSTEM_CONTEXT,
        messages=[{
            'role': 'user',
            'content': f"""Given the following GitHub issue, write the query you would pass to search_bugs if you encountered this bug in a developer's code. Just the query text, nothing else.

Title: {title}
Problem: {body[:500]}
Resolution: {resolution[:300]}"""
        }]
    )
    return msg.content[0].text.strip()

ground_truth = []
for numeric_id, issue_number, title, body, resolution, library in sampled:
    query = generate_query(title, body, resolution)
    ground_truth.append({
        'numeric_id': numeric_id,
        'issue_number': issue_number,
        'library': library,
        'title': title,
        'query': query
    })
    print(f'[{library}] #{issue_number}: {query}')


[flask] #1232: flask docs PDF zip download 404 broken links
[langchain] #9082: StructuredOutputParser invalid JSON parsing failure extra missing quotes
[fastapi] #3242: app.dependency_overrides get_db override_get_db not working pytest
[fastapi] #5493: uvicorn SSL client certificate not exposed in request
[fastapi] #5308: No 'Access-Control-Allow-Origin' header is present CORS error FastAPI
[pydantic] #5453: AnalyzedType model_rebuild update_forward_refs forward reference resolution
[requests] #5581: requests HTTP headers parsing error multiple response lines
[pydantic] #5182: `ImportError cannot import name BaseSettings from pydantic fastapi`
[next.js] #239: filesystem as API json folder REST API backend next.js
[requests] #6112: proxy authentication username password special characters encoding urllib.parse quote
[langchain] #2002: parquet file support SQL agent duckdb query
[sqlmodel] #319: specify SQL Server column data type sa_type dtype equivalent SQLModel
[flask] #1916: simplejs

In [6]:
# Load all embeddings from DB once
conn = get_conn()
cur = conn.cursor()
cur.execute('SELECT issue_id, embedding FROM embeddings')
all_embeddings = cur.fetchall()  # [(issue_id, embedding), ...]
cur.close()
conn.close()

print(f'Loaded {len(all_embeddings)} embeddings')

Loaded 46204 embeddings


In [7]:
# Run baseline search for each ground truth query
def search(query_text, top_k=3):
    q_emb = embed(query_text)
    scores = [(issue_id, cosine(q_emb, emb)) for issue_id, emb in all_embeddings]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [issue_id for issue_id, _ in scores[:top_k]], scores

results = []
for item in ground_truth:
    top3, scores = search(item['query'])
    rank = None
    for i, issue_id in enumerate(top3, 1):
        if issue_id == item['numeric_id']:
            rank = i
            break
    # Also check rank outside top 3
    full_rank = next((i+1 for i, (iid, _) in enumerate(scores) if iid == item['numeric_id']), None)
    results.append({
        **item,
        'top3': top3,
        'hit': rank is not None,
        'rank_in_top3': rank,
        'actual_rank': full_rank
    })
    status = f'HIT @{rank}' if rank else f'MISS (rank {full_rank})'
    print(f'[{item["library"]}] #{item["issue_number"]}: {status}')

[flask] #1232: MISS (rank 5)
[langchain] #9082: MISS (rank 7)
[fastapi] #3242: MISS (rank 26)
[fastapi] #5493: MISS (rank 6)
[fastapi] #5308: HIT @2
[pydantic] #5453: HIT @1
[requests] #5581: HIT @2
[pydantic] #5182: MISS (rank 28)
[next.js] #239: HIT @2
[requests] #6112: MISS (rank 4)
[langchain] #2002: HIT @1
[sqlmodel] #319: HIT @1
[flask] #1916: HIT @2
[python-dotenv] #5: HIT @2
[celery] #1925: HIT @1
[fastapi] #2948: HIT @1
[redis-py] #2563: HIT @1
[pyrefly] #595: MISS (rank 41)
[transformers] #7633: HIT @1
[langchain] #1968: HIT @2
[openai-python] #621: HIT @2
[tiktoken] #428: HIT @3
[openai-python] #240: HIT @2
[langchain] #13802: MISS (rank 12)
[psycopg2] #219: HIT @1
[transformers] #5142: MISS (rank 27)
[langchain] #9502: HIT @1
[flask] #4124: HIT @1
[next.js] #1033: HIT @1
[celery] #4288: HIT @1


In [8]:
# Metrics
hit_at_3 = sum(1 for r in results if r['hit']) / len(results)
mrr = sum(1 / r['actual_rank'] for r in results if r['actual_rank']) / len(results)

print(f'N = {len(results)}')
print(f'Hit@3:  {hit_at_3:.2%}')
print(f'MRR:    {mrr:.4f}')

misses = [r for r in results if not r['hit']]
print(f'\nMisses ({len(misses)}):')
for r in misses:
    print(f'  [{r["library"]}] #{r["issue_number"]} rank={r["actual_rank"]} — query: {r["query"]}')
    print(f'    title: {r["title"][:80]}')

N = 30
Hit@3:  70.00%
MRR:    0.5771

Misses (9):
  [flask] #1232 rank=5 — query: flask docs PDF zip download 404 broken links
    title: The 'download as PDF and zipped HTML' links on flask.pocoo.org are broken. 
  [langchain] #9082 rank=7 — query: StructuredOutputParser invalid JSON parsing failure extra missing quotes
    title: Issue: StructuredOutputParser failures due to invalid JSON
  [fastapi] #3242 rank=26 — query: app.dependency_overrides get_db override_get_db not working pytest
    title: app.dependency_overrides[get_db] = override_get_db is not working
  [fastapi] #5493 rank=6 — query: uvicorn SSL client certificate not exposed in request
    title: Obtain SSL client certificate from FastAPI request
  [pydantic] #5182 rank=28 — query: `ImportError cannot import name BaseSettings from pydantic fastapi`
    title: Pydantic import error in fastapi 
  [requests] #6112 rank=4 — query: proxy authentication username password special characters encoding urllib.parse quote
    titl